# EcoMind AI - Intelligent Waste Management Decision Agent

**Built for the AWS User Group Madurai Strands Decider Hackathon**

EcoMind AI classifies waste-management requests into four categories using Strands Decider 2B,
then routes to the right tools. Amazon Bedrock powers the conversational agent.

| Metric | Strands Decider 2B | LLM Baseline |
|---|---|---|
| Accuracy | 91.7% | 79.2% |
| Latency p50 | 1814 ms | 10684 ms |
| Speedup | 5.9x faster | baseline |
| Accuracy when confident | 94.7% | 79.2% |

**Tags**: #strands-decider-2b #policy

*Built with the AWS User Group Madurai Strands Decider hackathon kit.*

## 1. Setup

Install dependencies and import the EcoMind AI agent.

In [ ]:
import sys, ossys.path.insert(0, '..')from decider_kit.config import load_dotenvload_dotenv()print('DECIDER_MODE:', os.environ.get('DECIDER_MODE', 'local'))print('LLM_PROVIDER:', os.environ.get('LLM_PROVIDER', 'bedrock'))print('Setup complete.')

## 2. The #policy Decision Pattern

The reusable decision pattern in gent/decision.py uses Strands Decider 2Bto classify every request into one of four policy categories.

In [ ]:
from agent.decision import PolicyClassifier, CATEGORY_QUESTION, QUESTIONS, PATTERN_TAGprint('PATTERN_TAG:', PATTERN_TAG)print('Question type:', CATEGORY_QUESTION['type'])print('Options:')for k, v in CATEGORY_QUESTION['criteria'].items():    print(f'  {k}: {v}')

## 3. Test the Decider Classification

Send real requests to Strands Decider 2B and see the classification.

In [ ]:
from decider_kit import Deciderd = Decider()print('Decider hosted:', d.hosted)print('Decider URL:', d.url)print()test_messages = [    'A garbage bin near the school is overflowing and smells terrible!',    'Create a collection plan for the bins that need urgent attention.',    'Report a damaged bin in the residential area.',    'What types of waste go in the green recycling bins?',    'bin',]for msg in test_messages:    answers = d.ask(msg, QUESTIONS)    dec = answers['category']    print(f'Input:      {msg[:60]}')    print(f'Category:   {dec.label}')    print(f'Confidence: {dec.confidence:.3f}')    print(f'Low conf:   {dec.unsure(0.7)}')    print()

## 4. Test the Four Tools

Verify each tool with valid, invalid and edge-case inputs.

In [ ]:
import agent.tools as Tfn = lambda t: t.func if hasattr(t, 'func') else tget_bin = fn(T.get_bin_status)assess = fn(T.assess_collection_priority)plan = fn(T.create_collection_plan)report = fn(T.report_bin_issue)print('=== get_bin_status ===')print('Valid BIN-001:', get_bin('BIN-001')['bin_id'])print('Unknown BIN-999:', get_bin('BIN-999'))print('Empty:', get_bin(''))print('Case-insensitive bin-001:', get_bin('bin-001')['bin_id'])print()print('=== assess_collection_priority ===')r = assess(None)print('Total bins:', len(r['bins']))print('Top priority:', r['bins'][0]['bin_id'], 'score=', r['bins'][0]['priority_score'])print('Overflow bins:', sum(1 for b in r['bins'] if b['overflow']))print()print('=== create_collection_plan ===')p = plan(None)print('Recommendation:', p['recommendation'])print('Capacity used:', p['capacity_used'])print('Note:', p['note'])print('Plan:', [(b['rank'], b['bin_id']) for b in p['plan']])print()print('=== report_bin_issue ===')T.ISSUES.clear()r1 = report('BIN-007', 'Bin lid is cracked and cannot close')print('Report 1:', r1['report_id'], 'dup=', r1['duplicate'])r2 = report('BIN-007', 'Bin lid is cracked and cannot close')print('Report 2 (dup):', r2['report_id'], 'dup=', r2['duplicate'])r3 = report('BIN-999', 'Issue')print('Unknown bin:', r3)

## 5. Full Agent Chat

Test the complete HackathonAgent.chat() flow.

In [ ]:
from agent.agent import HackathonAgent, NAMEprint('Agent:', NAME)print()a = HackathonAgent()result = a.chat('A garbage bin near the school is almost overflowing. What should we do?')print('Reply:', result['reply'][:200])print('Decisions:', len(result['decisions']))dec = result['decisions'][0]print('Category:', dec['category'])print('Confidence:', round(dec['confidence'], 3))print('Verdict:', dec['verdict'])print('Latency:', result['latency_ms'], 'ms')

## 6. Simulated Bin Data

All data is clearly labelled as simulated.

In [ ]:
import jsonfrom pathlib import Pathbins = json.loads(Path('../agent/data/bins.json').read_text())config = json.loads(Path('../agent/data/config.json').read_text())print('Bins:', len(bins))print('Overflow threshold:', config['overflow_threshold'], '%')print('Vehicle capacity:', config['vehicle_capacity'])print('Data note:', config['data_note'])print()for b in bins:    status = 'OVERFLOW' if b['fill_level'] >= 90 else 'near' if b['fill_level'] >= 80 else 'ok'    issue = ' ISSUE:' + b['issue_status'] if b['issue_status'] != 'none' else ''    print(f'  {b["id"]}: {b["fill_level"]}% [{status}] {b["location"]}{issue}')

## 7. Benchmark Results

Real results from 24 labelled cases.

In [ ]:
from pathlib import Pathresults = Path('../submission/bench/results.md').read_text()print(results)

## 8. Low-Confidence Handling

When the decider is unsure (confidence < 0.7), the agent returns clarification.

In [ ]:
sys.path.insert(0, '../tests')from fakes import FakeDeciderfrom decider_kit.client import Decisionfrom agent.decision import PolicyClassifierclass LowConfDecider:    last_latency_ms = 10.0    def ask(self, state, questions, images=None):        probs = {'urgent_overflow': 0.35, 'routine_collection': 0.25,                 'bin_issue': 0.25, 'general_information': 0.15}        return {'category': Decision.parse(            {'type': 'choice', 'choice': 'urgent_overflow',             'probabilities': probs, 'confidence': 0.35})}clf = PolicyClassifier(LowConfDecider())result = clf.classify('something vague about a bin')print('Category:', result['category'])print('Confidence:', result['confidence'])print('Low confidence:', result['low_confidence'])print('Verdict:', result['_record']['verdict'])print('-> Agent would ask for clarification, NOT act as emergency')

## 9. Run the Test Suite

36 unit tests covering all tools, classifier and agent contract.

In [ ]:
import subprocessr = subprocess.run([sys.executable, '-m', 'pytest', '../tests/test_ecomind.py', '-q', '--tb=short'],                   capture_output=True, text=True, cwd='..')print(r.stdout)if r.stderr: print(r.stderr)

---

**Built with the AWS User Group Madurai Strands Decider hackathon kit.**

GitHub: https://github.com/rishirishitha047-wq/Smart-Waste-Decision-Agent